# FireCrawl

### What is web scraping?

**Web scraping** means having a program automatically visit web pages and pull out the content (text, links, images, etc.) instead of a human copy-pasting it. It's useful when you want data that lives on websites — news, docs, product listings — in a structured form you can use in code (e.g. feed it to an LLM, save it to a file, put it in a database).

The hard parts of scraping yourself: pages are messy HTML, some need JavaScript to render, some block bots, and PDFs/images need special handling. **Firecrawl** is a hosted API that does all of that for you: you give it a URL (or a search query), it fetches the page(s), and hands you back clean text (usually Markdown) plus metadata.

Firecrawl has three main things you can ask it to do, and this notebook has one section for each:
- **Search** — search the web like a search engine, optionally fetching ("scraping") each result's content too.
- **Crawl** — start at one URL and automatically follow links to fetch *many* pages from that site.
- **Scrape** — fetch and clean up the content of *one specific* URL you already know.

Firecrawl is a paid API: every request costs "credits" depending on how much it scrapes. You'll see `credits_used` printed in a few places below so you can see what things cost.

In [ ]:
# --- Setup: imports and API key ---
import requests                  # lets Python make HTTP calls (GET/POST) to web APIs
import os                        # standard library, used here to read environment variables
from dotenv import load_dotenv   # reads a local .env file and loads it into the environment

load_dotenv()  # without this, os.environ wouldn't see what's written in your .env file

# Your Firecrawl API key should live in .env as FIRECRAWL_API_KEY=... — never hard-code
# API keys directly in a notebook, since notebooks often get shared/committed to git.
FIRECRAWL_API = os.environ.get("FIRECRAWL_API_KEY")
print("Key loaded:", bool(FIRECRAWL_API), "len:", len(FIRECRAWL_API) if FIRECRAWL_API else 0)

# --- Firecrawl's /search endpoint ---
# This endpoint works like a search engine: give it a query, get back result URLs.
# With `scrapeOptions` set, it also scrapes each result's page content for you,
# so you get full page text back, not just links.
url = "https://api.firecrawl.dev/v2/search"

payload = {
  "query": "what are the latest news about Jensen Huang?",  # what to search for
  "sources": [
    "web"              # search regular web pages (other sources include "news", "images")
  ],
  "categories": [],
  "limit": 3,            # how many search results to return
  "scrapeOptions": {      # ask Firecrawl to also fetch each result page's content
    "onlyMainContent": True,  # strip nav bars/ads/footers, keep just the article body
    "maxAge": 172800000,      # (milliseconds, = 48h) reuse a recent cached scrape if one exists,
                               # instead of re-scraping — saves time and credits
    "parsers": [
      "pdf"              # if a result is a PDF, extract its text too
    ],
    "formats": []         # which content formats to return (e.g. "markdown", "html") — none here,
                           # we only care about the metadata for this first example
  }
}

headers = {
    "Authorization": f"Bearer {FIRECRAWL_API}",  # Firecrawl expects "Bearer <your_api_key>"
    "Content-Type": "application/json"
}

# requests.post() sends the HTTP request and waits (blocks) until Firecrawl responds.
web_response = requests.post(url, json=payload, headers=headers)

print("Status:", web_response.status_code)   # HTTP status: 200 = success, 401 = bad key,
                                              # 429 = rate limited, 5xx = server error
print("Body:", web_response.text[:1000])      # raw text of the response — useful when debugging,
                                              # since .json() below would just error on a bad body
web_response.raise_for_status()               # raises an exception here if status means failure,
                                              # instead of silently continuing with no data
print(web_response.json())                    # parse the JSON response body into a Python dict

# --- Alexandria tools index ---
# "alexandria" is a different, separate Firecrawl search source: a curated index of
# known tools/APIs/datasets, rather than the open web. Same endpoint, different `sources`.
alexandria_response = requests.post(
    "https://api.firecrawl.dev/v2/search",
    json={
        "query": "your search query",
        "sources": ["alexandria"],
        "limit": 10,
    },
    headers={"Authorization": f"Bearer {FIRECRAWL_API}", "Content-Type": "application/json"},
)
print("Status:", alexandria_response.status_code)
print("Body:", alexandria_response.text[:1000])
alexandria_response.raise_for_status()
print(alexandria_response.json())

### Exploring the search response

`web_response` is a `requests.Response` object (the raw HTTP reply) — it's not the data itself.
Calling `.json()` on it parses the response **body** (which is JSON text) into a Python `dict`.
Every cell below just digs into that dict to see what Firecrawl sent back:

- `success` — did the request succeed (bool)
- `data` — the actual results (what we care about)
- `creditsUsed` — how many Firecrawl credits this request cost you
- `id` — an id for this request, useful if you need to contact Firecrawl support about it

In [ ]:
# Top-level keys of the parsed response dict (see markdown above)
web_response.json().keys()

In [ ]:
# How many Firecrawl credits this one search request cost
web_response.json()["creditsUsed"]

In [ ]:
# Save the parsed dict so we don't have to call .json() again (it re-reads the response each time)
result = web_response.json()
# `data` holds the actual search results, grouped by source — we only asked for "web"
result["data"].keys()

In [ ]:
# Should be 3 — matches the "limit": 3 we set in the payload
len(result["data"]["web"])

In [ ]:
# Each item here is one search result: typically a title, url, and a short description/snippet
result["data"]["web"]

In [ ]:
# Persist the results to disk as JSON, so you have them saved without re-calling the API
import json

with open("jensen_huang.json", "w", encoding="utf-8") as f:
    json.dump(result["data"]["web"], f, indent=2)

## Crawl several pages

**Crawling** is different from searching: you give Firecrawl one starting URL on a site, and it
automatically discovers and fetches *multiple pages* from that same site (following links/sitemap),
instead of you listing every URL yourself.

From here on we switch from raw `requests` calls to the **`firecrawl-py`** SDK (`pip install firecrawl-py`).
It's just a Python wrapper around the same HTTP API — it saves you from building the JSON payloads
by hand and gives back typed Python objects instead of raw dicts.

In [8]:
# Run once if firecrawl-py isn't installed yet
# %pip install firecrawl-py

In [ ]:
from pathlib import Path
from firecrawl import Firecrawl              # the SDK client
from firecrawl.v2.types import ScrapeOptions  # typed settings object for how to scrape each page

# The SDK client wraps your API key and knows how to call every Firecrawl endpoint.
app = Firecrawl(api_key=FIRECRAWL_API)

# Same idea as the raw `scrapeOptions` dict from the search example, but as a typed object —
# these settings apply to every page the crawl visits.
scrape_opts = ScrapeOptions(
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"],
)

data_path = Path("myh_data")

# Crawling costs 1 credit per page it fetches, so guard against re-running this cell by
# accident: if we already saved pages from a previous run, skip calling the API again.
if data_path.exists() and any(data_path.iterdir()):
    print(f"Skipping crawl: {data_path} already has {len(list(data_path.iterdir()))} file(s).")
    crawl_result = None
else:
    crawl_result = app.crawl(
        "https://www.myh.se/yrkeshogskolan",  # the starting URL
        sitemap="include",        # use the site's sitemap.xml to help discover pages, if present
        crawl_entire_domain=False,  # stay under this path, don't wander off to the whole domain
        limit=10,                 # fetch at most 10 pages
        scrape_options=scrape_opts,
    )

### Exploring the crawl result

Unlike the raw `requests` example, the SDK gives back a Python **object**, not a dict — so we use
`.attribute` instead of `["key"]`. `crawl_result.__dict__.keys()` lists its attributes, same idea
as `.keys()` did for a dict. (If `crawl_result` is `None` because the crawl was skipped above,
these cells will error — that's expected, it just means there's nothing new to inspect.)

In [ ]:
# Attributes available on the crawl result object
crawl_result.__dict__.keys()

In [ ]:
# Total credits this crawl cost (roughly 1 per page fetched)
crawl_result.credits_used

In [ ]:
# Number of pages the crawl actually fetched (capped by our limit=10)
len(crawl_result.data)

In [ ]:
# Each crawled page is a "Document" object with these fields — `markdown` is the cleaned
# page content, `metadata` has things like the page title and URL
crawl_result.data[0].__dict__.keys()

In [ ]:
# Save every crawled page's markdown content to its own file, named after the page title
if crawl_result is None:
    print(f"Nothing to save: using existing files in {data_path}")
else:
    data_path.mkdir(exist_ok=True)
    for page in crawl_result.data:
        title = page.metadata.title or "untitled"
        # Filenames can't contain characters like \ / : * ? " < > | — strip anything that
        # isn't a letter, digit, space, underscore or hyphen to avoid an OSError on write
        safe_title = "".join(c for c in title if c.isalnum() or c in (" ", "_", "-")).strip()
        (data_path / f"{safe_title}.md").write_text(page.markdown, encoding="utf-8")

## Scrape a single page

**Scraping** is the simplest Firecrawl operation: you already know the exact URL, and you just
want its content — no searching, no following links to other pages.

In [ ]:
# app.scrape() takes the scrape settings as individual keyword arguments directly —
# unlike app.crawl(), it does NOT accept a `scrape_options=ScrapeOptions(...)` object,
# so we pass each field from scrape_opts separately here.
page_data = app.scrape(
    "https://pydantic.dev/docs/ai/evals/evals/",
    only_main_content=scrape_opts.only_main_content,
    max_age=scrape_opts.max_age,
    parsers=scrape_opts.parsers,
    formats=scrape_opts.formats,
)

In [ ]:
# A single scrape returns one Document object directly (not a list, like crawl_result.data was)
page_data.__dict__.keys()

In [ ]:
# The page's <title> — Firecrawl extracts standard page metadata for you automatically
page_data.metadata.title

In [ ]:
# The page's meta description tag
page_data.metadata.description

In [ ]:
# All the metadata Firecrawl extracted: title, description, OpenGraph tags, status_code,
# how many credits this scrape cost, whether it was served from cache, etc.
page_data.metadata.__dict__.keys()

In [ ]:
# The final URL that was actually scraped (useful if the original URL redirected somewhere)
page_data.metadata.source_url

In [ ]:
# Which type of proxy Firecrawl used to fetch the page (e.g. "basic" vs a stealthier one
# for sites that block simple bots)
page_data.metadata.proxy_used

In [ ]:
# NOTE: this errors on Windows with `OSError: Invalid argument`!
# page_data.metadata.title is something like "Pydantic Evals | Pydantic Docs" — the "|"
# character isn't allowed in a Windows filename (same problem the crawl-saving cell above
# avoided by stripping unsafe characters). See the fixed version further down, which
# replaces "|" before writing the file.
Path(f"{page_data.metadata.title}.md").write_text(page_data.markdown, encoding="utf-8")

# Webscraping (specific page)

This repeats the single-page scrape above, a bit more cleanly: `Firecrawl()` is created without
re-passing the API key explicitly (the SDK also reads `FIRECRAWL_API_KEY` from the environment
on its own, since `load_dotenv()` already ran earlier), and the filename bug from above is fixed.

In [ ]:
from firecrawl import Firecrawl

# No api_key passed here — the SDK falls back to reading FIRECRAWL_API_KEY from the
# environment itself, which load_dotenv() already loaded earlier in the notebook.
app = Firecrawl()

data = app.scrape(
    "https://pydantic.dev/docs/ai/evals/evals/",
    only_main_content=True,   # drop nav/ads/footers
    max_age=172800000,        # reuse a cached scrape (48h) if available
    parsers=["pdf"],          # extract text if the URL turns out to be a PDF
    formats=["markdown"],     # return the content as cleaned-up Markdown
)

In [ ]:
# Same Document fields we saw before: markdown content, html, metadata, links, images, etc.
data.__dict__.keys()

In [ ]:
# Pull the metadata object out into its own variable so the cells below read a bit cleaner
metadata = data.metadata
metadata.__dict__.keys()

In [ ]:
# The `{var = }` f-string syntax prints both the variable name and its value — handy for
# quickly inspecting several values without writing print("label:", value) each time
print(f"{metadata.title = }")
print(f"{metadata.description = }")
print(f"{metadata.url = }")
print(f"{metadata.language = }")
print(f"{metadata.keywords = }")
print(f"{metadata.robots = }")

In [ ]:
# Fixed version of the earlier filename bug: strip the "|" character before using the
# title as a filename, so this works on Windows too
with open(f"{metadata.title.replace("|", "")}.md", "w", encoding="utf-8") as f:
    f.write(data.markdown)

## Key takeaways

- **Search** = find pages matching a query (optionally scraping each result too).
- **Crawl** = start at one URL, auto-follow links to fetch many pages from that site.
- **Scrape** = fetch and clean up exactly one URL you already know.
- Firecrawl handles the messy parts of scraping (JS rendering, bot-blocking, PDFs) and gives you
  back clean Markdown + metadata instead of raw HTML.
- Every call costs **credits** — `maxAge` caching and the "skip if already saved" check in the
  crawl cell both exist to avoid paying for (and waiting on) the same scrape twice.
- The raw-`requests` version (top of the notebook) and the `firecrawl-py` SDK version (rest of
  the notebook) talk to the *same* API — the SDK just saves you from building JSON payloads and
  gives typed objects (`.attribute`) back instead of plain dicts (`["key"]`).